# 03 — Strategy Development

Define a strategy, sweep parameters over a grid, and get Claude's
interpretation of parameter sensitivity.

**Cost per full run**: ~£0.02 (one Claude API call at ~1k tokens).

Runs without an API key — all charts render; interpretations show unavailable.

### Path setup

`sys.path.insert(0, "..")` makes `src` importable from `notebooks/`.

In [ ]:
import sys
sys.path.insert(0, "..")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.utils.config import load_config
from src.services.data_service import DataService
from src.features.pipeline import FeaturePipeline
from src.backtest.engine import BacktestEngine
from src.backtest.strategy import MeanReversionStrategy
from src.notebooks.formatters import format_backtest_metrics, display_interpretation
from src.notebooks.research_log import ResearchLog

In [ ]:
config = load_config()
data_service = DataService(config)
pipeline = FeaturePipeline(config)
engine = BacktestEngine(config)
research_log = ResearchLog(config)

ticker = config["universe"]["tickers"][0]
print(f"Strategy development for: {ticker}")

## Section 1 — Define Strategy and Baseline Backtest

In [ ]:
# Load data and generate features.
ohlcv = data_service.get_price_data(ticker)
features = pipeline.generate(ohlcv)

# Baseline: default RSI mean-reversion strategy.
baseline_strat = MeanReversionStrategy("baseline_mean_rev")
baseline_result = engine.run(baseline_strat, ohlcv, features)

print(f"Baseline Sharpe: {baseline_result.metrics.get('sharpe', 'N/A'):.4f}")
print(f"Baseline Max DD: {baseline_result.metrics.get('max_drawdown', 'N/A'):.4f}")
print(f"Total trades: {baseline_result.metrics.get('total_trades', 0)}")

## Section 2 — Parameter Sweep

In [ ]:
# Sweep RSI oversold/overbought thresholds.
oversold_range = range(20, 40, 5)    # 20, 25, 30, 35
overbought_range = range(65, 85, 5)  # 65, 70, 75, 80

sweep_results = []

for os_thresh in oversold_range:
    for ob_thresh in overbought_range:
        strat_config = {
            "oversold_threshold": os_thresh,
            "overbought_threshold": ob_thresh,
        }
        strat = MeanReversionStrategy(
            f"mr_{os_thresh}_{ob_thresh}", config=strat_config
        )
        try:
            result = engine.run(strat, ohlcv, features)
            sharpe = result.metrics.get("sharpe", 0.0)
        except Exception:
            sharpe = 0.0

        sweep_results.append({
            "oversold": os_thresh,
            "overbought": ob_thresh,
            "sharpe": round(sharpe, 4),
        })

sweep_df = pd.DataFrame(sweep_results)
print(f"Completed {len(sweep_df)} parameter combinations.")
sweep_df.sort_values("sharpe", ascending=False).head(5)

In [ ]:
# Sharpe heatmap.
pivot = sweep_df.pivot(index="oversold", columns="overbought", values="sharpe")

fig = px.imshow(
    pivot,
    text_auto=".2f",
    title="Parameter Sensitivity: Sharpe Ratio",
    labels={"x": "Overbought Threshold", "y": "Oversold Threshold", "color": "Sharpe"},
    color_continuous_scale="RdYlGn",
)
fig.show()

## Section 3 — Claude Interpretation

In [ ]:
# Prepare data for Claude.
best_row = sweep_df.loc[sweep_df["sharpe"].idxmax()]
worst_row = sweep_df.loc[sweep_df["sharpe"].idxmin()]

comparison_data = {
    "ticker": ticker,
    "strategy": "RSI Mean Reversion",
    "parameter_grid": {
        "oversold": list(oversold_range),
        "overbought": list(overbought_range),
    },
    "total_combinations": len(sweep_df),
    "best_params": {
        "oversold": int(best_row["oversold"]),
        "overbought": int(best_row["overbought"]),
        "sharpe": float(best_row["sharpe"]),
    },
    "worst_params": {
        "oversold": int(worst_row["oversold"]),
        "overbought": int(worst_row["overbought"]),
        "sharpe": float(worst_row["sharpe"]),
    },
    "sharpe_std": round(float(sweep_df["sharpe"].std()), 4),
    "sharpe_mean": round(float(sweep_df["sharpe"].mean()), 4),
    "all_results": sweep_results,
}

print(f"Best: oversold={best_row['oversold']}, overbought={best_row['overbought']}, sharpe={best_row['sharpe']}")
print(f"Worst: oversold={worst_row['oversold']}, overbought={worst_row['overbought']}, sharpe={worst_row['sharpe']}")

In [ ]:
try:
    from src.notebooks.claude_interpreter import QuantInterpreter

    interpreter = QuantInterpreter(config)
    interpretation = interpreter.interpret("strategy_comparison", comparison_data)
except (EnvironmentError, ImportError) as e:
    print(f"Claude interpretation unavailable: {e}")
    interpretation = {
        "summary": "Interpretation unavailable (no API key set)",
        "observations": [],
        "warnings": ["Set ANTHROPIC_API_KEY to enable interpretations"],
        "suggestions": [],
        "confidence": "low",
    }

display_interpretation(interpretation)

In [ ]:
research_log.log_entry(
    notebook="03_strategy_development",
    task="strategy_comparison",
    data_summary=comparison_data,
    interpretation=interpretation,
)
print("Entry logged.")